In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = "/content/drive/MyDrive/Thesis/ai_model_predictions.csv"

cols = [
    "doc_id",
    "flag_patent",
    "pub_dt",
    "predict50_any_ai",
    "predict86_any_ai",
    "predict93_any_ai"
]

df_aipd = pd.read_csv(
    path,
    usecols=cols,
    parse_dates=["pub_dt"]
)

df_aipd = df_aipd[df_aipd["pub_dt"] >= "2016-01-01"]

# keep only rows that are actual patents
df_aipd = df_aipd[df_aipd["flag_patent"] == 1]

df_aipd.head()

/tmp/ipykernel_6847/788854458.py:12: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_aipd = pd.read_csv(


,doc_id,flag_patent,pub_dt,predict50_any_ai,predict86_any_ai,predict93_any_ai
0,10000000,1,2018-06-19,0,0,0
1,10000001,1,2018-06-19,0,0,0
2,10000002,1,2018-06-19,1,1,1
3,10000003,1,2018-06-19,0,0,0
4,10000004,1,2018-06-19,0,0,0


In [ ]:
path = "/content/drive/MyDrive/Thesis/ai_model_predictions.csv"
df = pd.read_csv(path, nrows=0)

print(df.columns)

Index(['doc_id', 'flag_patent', 'pub_dt', 'appl_id', 'predict50_any_ai',
       'predict86_any_ai', 'predict93_any_ai', 'predict50_ml', 'predict86_ml',
       'predict93_ml', 'ai_score_ml', 'predict50_evo', 'predict86_evo',
       'predict93_evo', 'ai_score_evo', 'predict50_nlp', 'predict86_nlp',
       'predict93_nlp', 'ai_score_nlp', 'predict50_speech', 'predict86_speech',
       'predict93_speech', 'ai_score_speech', 'predict50_vision',
       'predict86_vision', 'predict93_vision', 'ai_score_vision',
       'predict50_planning', 'predict86_planning', 'predict93_planning',
       'ai_score_planning', 'predict50_kr', 'predict86_kr', 'predict93_kr',
       'ai_score_kr', 'predict50_hardware', 'predict86_hardware',
       'predict93_hardware', 'ai_score_hardware'],
      dtype='object')


In [ ]:
print(df_aipd.shape)
print(df_aipd["flag_patent"].value_counts())

(2602117, 6)
flag_patent
1    2602117
Name: count, dtype: int64


In [ ]:
# data types
df_aipd.dtypes

,0
doc_id,object
flag_patent,int64
pub_dt,datetime64[ns]
predict50_any_ai,int64
predict86_any_ai,int64
predict93_any_ai,int64


In [ ]:
path = "/content/drive/MyDrive/Thesis/g_patent.tsv"
df_patent = pd.read_csv(path, sep="\t", nrows=1000)
df_patent.head()

,patent_id,patent_type,patent_date,patent_title,wipo_kind,num_claims,withdrawn,filename
0,10000000,utility,2018-06-19,Coherent LADAR using intra-pixel quadrature de...,B2,20,0,ipg180619.xml
1,10000001,utility,2018-06-19,Injection molding machine and mold thickness c...,B2,12,0,ipg180619.xml
2,10000002,utility,2018-06-19,Method for manufacturing polymer film and co-e...,B2,9,0,ipg180619.xml
3,10000003,utility,2018-06-19,Method for producing a container from a thermo...,B2,18,0,ipg180619.xml
4,10000004,utility,2018-06-19,"Process of obtaining a double-oriented film, c...",B2,6,0,ipg180619.xml


In [ ]:
print(df_patent['patent_type'].value_counts())

patent_type
utility    1000
Name: count, dtype: int64


In [ ]:
path = "/content/drive/MyDrive/Thesis/g_assignee_disambiguated.tsv"

cols = [
    "patent_id",
    "assignee_sequence",
    "assignee_id",
    "disambig_assignee_organization",
    "assignee_type"
]

df_assignee = pd.read_csv(
    path,
    sep="\t",
    usecols=cols,
    dtype={
        "patent_id": "string",
        "assignee_sequence": "int8",
        "assignee_id": "string",
        "disambig_assignee_organization": "string",
        "assignee_type": "float32"
    }
)

df_assignee.head()

,patent_id,assignee_sequence,assignee_id,disambig_assignee_organization,assignee_type
0,4488683,0,1ea59611-1100-41e1-8805-48bb4d364f0a,Metal Works Ramat David,3.0
1,11872626,0,a697843f-fe51-4800-be08-e9ea92ffe21d,"Divergent Technologies, Inc.",2.0
2,5856666,0,6807e85b-863f-4c80-8a1d-436da8608d50,U.S. Philips Corporation,2.0
3,5204210,0,784b5ac7-b74d-422e-9cc1-922f47801deb,Xerox Corporation,2.0
4,5302149,1,ad38a161-9f04-4f06-80c9-3ee8305d23c6,COMMONWEALTH SCIENTIFIC AND INDUSTRIAL RESEARC...,7.0


In [ ]:
df_assignee.shape

(8660702, 5)

In [ ]:
df_aipd["doc_id"] = df_aipd["doc_id"].astype(str).str.strip()
df_patent["patent_id"] = df_patent["patent_id"].astype(str).str.strip()
df_assignee["patent_id"] = df_assignee["patent_id"].astype(str).str.strip()

print(df_aipd["doc_id"].dtype)
print(df_patent["patent_id"].dtype)
print(df_assignee["patent_id"].dtype)

object
object
object


In [ ]:
df_ai_assignee = df_aipd.merge(
    df_assignee,
    left_on="doc_id",
    right_on="patent_id",
    how="left"
)

# keep US corporate assignees only
df_ai_assignee = df_ai_assignee[
    df_ai_assignee["assignee_type"].isin([2])
].copy()

df_ai_assignee = df_ai_assignee.rename(
    columns={"disambig_assignee_organization": "firm_name"}
)

df_ai_assignee.head() # patient-level dataset, each row is a patent

,doc_id,flag_patent,pub_dt,predict50_any_ai,predict86_any_ai,predict93_any_ai,patent_id,assignee_sequence,assignee_id,firm_name,assignee_type
0,10000000,1,2018-06-19,0,0,0,10000000,0.0,9e8542bf-1e3f-45ae-8584-6a6bbdce5021,Raytheon Company,2.0
7,10000007,1,2018-06-19,0,0,0,10000007,0.0,a04d7440-299a-488e-8f04-0ef6c6350629,Milwaukee Electric Tool Corporation,2.0
8,10000008,1,2018-06-19,0,0,0,10000008,0.0,d2f2f85e-0193-4f4f-a1c0-1bed52e75c1b,"Alex Toys, LLC",2.0
10,10000010,1,2018-06-19,0,0,0,10000010,0.0,784b5ac7-b74d-422e-9cc1-922f47801deb,Xerox Corporation,2.0
11,10000011,1,2018-06-19,0,0,0,10000011,0.0,57e6f3f6-e23e-4dac-9930-5d123ea131c9,"MARKFORGED, INC",2.0


In [ ]:
start_date = df_ai_assignee["pub_dt"].min()
end_date = df_ai_assignee["pub_dt"].max()

print(f"Date range: {start_date} to {end_date}")

Date range: 2016-01-05 00:00:00 to 2023-12-26 00:00:00


In [ ]:
df_ai_assignee.shape

(1200767, 11)

In [ ]:
import pandas as pd

# make sure pub_dt is datetime
df_ai_assignee["pub_dt"] = pd.to_datetime(df_ai_assignee["pub_dt"])

# step 1: create quarter variable from publication date
df_ai_assignee["quarter"] = df_ai_assignee["pub_dt"].dt.to_period("Q")

# step 2: aggregate to firm-quarter AI patents
firm_quarter_ai = (
    df_ai_assignee
    .groupby(["firm_name", "quarter"])["predict86_any_ai"]
    .sum()
    .reset_index()
    .rename(columns={"predict86_any_ai": "ai_patents"})
)

print("firm-quarter AI patents:")
print(firm_quarter_ai.head())

# step 3: calculate firm-quarter total patents
firm_quarter_total = (
    df_ai_assignee
    .groupby(["firm_name", "quarter"])
    .size()
    .reset_index(name="total_patents")
)

print("firm-quarter total patents:")
print(firm_quarter_total.head())

# step 4: merge AI patents and total patents
firm_quarter = firm_quarter_total.merge(
    firm_quarter_ai,
    on=["firm_name", "quarter"],
    how="left"
)

# step 5: fill missing AI patent counts with 0
firm_quarter["ai_patents"] = firm_quarter["ai_patents"].fillna(0)

# optional: convert to integer if you want clean counts
firm_quarter["ai_patents"] = firm_quarter["ai_patents"].astype(int)

# step 6: calculate AI share
firm_quarter["ai_share"] = (
    firm_quarter["ai_patents"] / firm_quarter["total_patents"]
)

# step 7: sort by firm and quarter
firm_quarter = (
    firm_quarter
    .sort_values(["firm_name", "quarter"])
    .reset_index(drop=True)
)

print("final firm-quarter dataset:")
print(firm_quarter.head())

# optional: if you want quarter shown as string like '2018Q2'
firm_quarter["quarter_str"] = firm_quarter["quarter"].astype(str)

# optional: inspect
print(firm_quarter[["firm_name", "quarter", "quarter_str", "total_patents", "ai_patents", "ai_share"]].head())

firm-quarter AI patents:
                              firm_name quarter  ai_patents
0                  #1 A LifeSafer, Inc.  2019Q4           1
1  (E)Mission Control Technologies, LLC  2016Q4           0
2                    (WE) FOR DOGS, LLC  2017Q4           0
3                    (WE) FOR DOGS, LLC  2019Q1           0
4                        .DECIMAL, INC.  2016Q2           0
firm-quarter total patents:
                              firm_name quarter  total_patents
0                  #1 A LifeSafer, Inc.  2019Q4              1
1  (E)Mission Control Technologies, LLC  2016Q4              1
2                    (WE) FOR DOGS, LLC  2017Q4              1
3                    (WE) FOR DOGS, LLC  2019Q1              1
4                        .DECIMAL, INC.  2016Q2              1
final firm-quarter dataset:
                              firm_name quarter  total_patents  ai_patents  \
0                  #1 A LifeSafer, Inc.  2019Q4              1           1   
1  (E)Mission Control Tec

In [ ]:
print("rows:", firm_quarter.shape)
print("firms:", firm_quarter["firm_name"].nunique())
print("quarters:", firm_quarter["quarter"].nunique())

print("AI patent distribution:", firm_quarter["ai_patents"].describe())

print("AI share distribution:", firm_quarter["ai_share"].describe())

rows: (291775, 6)
firms: 78921
quarters: 32
AI patent distribution: count    291775.000000
mean          1.147456
std          14.825408
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max        1759.000000
Name: ai_patents, dtype: float64
AI share distribution: count    291775.000000
mean          0.207884
std           0.379239
min           0.000000
25%           0.000000
50%           0.000000
75%           0.187500
max           1.000000
Name: ai_share, dtype: float64


In [ ]:
top_firms = (
    firm_quarter
    .groupby("firm_name")["ai_patents"]
    .sum()
    .sort_values(ascending=False)
)

# print("top firms:", top_firms.head(1000))

# top_names = top_firms.head(1000).index

# firm_quarter[firm_quarter["firm_name"].isin(top_names)] \
#     .sort_values(["firm_name", "quarter"])

# top_1000_list = top_firms.head(1000).index.tolist()

# print(top_1000_list)



print("top firms:", top_firms.head(500))

top_names = top_firms.head(500).index

firm_quarter[firm_quarter["firm_name"].isin(top_names)] \
    .sort_values(["firm_name", "quarter"])

top_500_list = top_firms.head(500).index.tolist()

print(top_500_list)

pd.Series(top_500_list, name="firm_name") \
    .to_csv("/content/drive/MyDrive/Thesis/top_500_ai_firms.csv", index=False)

top firms: firm_name
International Business Machines Corporation           35067
MICROSOFT TECHNOLOGY LICENSING, LLC                   12810
Google LLC                                            11061
AMAZON TECHNOLOGIES, INC.                              9897
Intel Corporation                                      7782
                                                      ...  
Netronome Systems, Inc.                                  66
SIEMENS PRODUCT LIFECYCLE MANAGEMENT SOFTWARE INC.       65
Rosemount Aerospace Inc.                                 65
Quantum Corporation                                      65
Kepler Computing Inc.                                    65
Name: ai_patents, Length: 500, dtype: int64
['International Business Machines Corporation', 'MICROSOFT TECHNOLOGY LICENSING, LLC', 'Google LLC', 'AMAZON TECHNOLOGIES, INC.', 'Intel Corporation', 'Apple Inc.', 'Oracle International Corporation', 'QUALCOMM Incorporated', 'AT&T Intellectual Property I, L.P.', 'EMC IP Hol

In [ ]:
# using current top_500_list to filter
top_500_firm_quarter = (
    firm_quarter[firm_quarter["firm_name"].isin(top_500_list)]
    [["firm_name", "quarter", "quarter_str", "total_patents", "ai_patents", "ai_share"]]
    .sort_values(["firm_name", "quarter"])
    .reset_index(drop=True)
)

print("Top 500 firm-quarter panel:")
print(top_500_firm_quarter.head(20))

print("Number of rows:", top_500_firm_quarter.shape[0])
print("Number of unique firms:", top_500_firm_quarter["firm_name"].nunique())

save_path = "/content/drive/MyDrive/Thesis/top_500_firm_quarter_panel.csv"
top_500_firm_quarter.to_csv(save_path, index=False)

print("Saved to:", save_path)

Top 500 firm-quarter panel:
                           firm_name quarter quarter_str  total_patents  \
0   3M INNOVATIVE PROPERTIES COMPANY  2016Q1      2016Q1            145   
1   3M INNOVATIVE PROPERTIES COMPANY  2016Q2      2016Q2            135   
2   3M INNOVATIVE PROPERTIES COMPANY  2016Q3      2016Q3            134   
3   3M INNOVATIVE PROPERTIES COMPANY  2016Q4      2016Q4            114   
4   3M INNOVATIVE PROPERTIES COMPANY  2017Q1      2017Q1            137   
5   3M INNOVATIVE PROPERTIES COMPANY  2017Q2      2017Q2            128   
6   3M INNOVATIVE PROPERTIES COMPANY  2017Q3      2017Q3            152   
7   3M INNOVATIVE PROPERTIES COMPANY  2017Q4      2017Q4            173   
8   3M INNOVATIVE PROPERTIES COMPANY  2018Q1      2018Q1            136   
9   3M INNOVATIVE PROPERTIES COMPANY  2018Q2      2018Q2            150   
10  3M INNOVATIVE PROPERTIES COMPANY  2018Q3      2018Q3            142   
11  3M INNOVATIVE PROPERTIES COMPANY  2018Q4      2018Q4            136 

In [ ]:
import pandas as pd

# =========================
# 1. read ticker mapping file
# =========================
path = "/content/drive/MyDrive/Thesis/top_500_ai_firms_with_ticker.xlsx"
sheet_name = "filtered us listed company"

ticker_map = pd.read_excel(path, sheet_name=sheet_name)

print("ticker_map shape:", ticker_map.shape)
print("\nticker_map columns:")
print(ticker_map.columns.tolist())
print("\nticker_map head:")
print(ticker_map.head())

# =========================
# 2. keep only needed columns
# =========================
ticker_map = ticker_map[["original_patent_file_name", "ticker"]].copy()

# optional: drop duplicate mapping rows
ticker_map = ticker_map.drop_duplicates(subset=["original_patent_file_name"])

print("\nticker_map shape after keeping needed columns:", ticker_map.shape)
print("\nticker_map head after cleanup:")
print(ticker_map.head())

# =========================
# 3. merge with firm_quarter
# =========================
firm_quarter_merged = firm_quarter.merge(
    ticker_map,
    left_on="firm_name",
    right_on="original_patent_file_name",
    how="left"
)

# =========================
# 4. keep only rows with non-null ticker
# =========================
firm_quarter_merged = firm_quarter_merged[firm_quarter_merged["ticker"].notna()].copy()

# =========================
# 5. keep final columns in requested order
# =========================
firm_quarter_merged = firm_quarter_merged[
    ["firm_name", "ticker", "quarter", "quarter_str", "total_patents", "ai_patents", "ai_share"]
].copy()

# =========================
# 6. inspect results
# =========================
print("\nmerged shape after dropping null tickers:", firm_quarter_merged.shape)
print("\nmerged head:")
print(firm_quarter_merged.head())

print("\nmerged info:")
firm_quarter_merged.info()

print("\nnon-null ticker row count:", firm_quarter_merged["ticker"].notna().sum())
print("unique ticker count:", firm_quarter_merged["ticker"].nunique())
print("unique firm_name count:", firm_quarter_merged["firm_name"].nunique())

# optional: check a few matched rows
print("\nmatched sample:")
print(firm_quarter_merged.head(10))

# =========================
# 7. save to csv
# =========================
output_path = "/content/drive/MyDrive/Thesis/firm_quarter_with_ticker_only.csv"
firm_quarter_merged.to_csv(output_path, index=False)

print(f"\nSaved merged file to: {output_path}")

ticker_map shape: (173, 6)

ticker_map columns:
['original_patent_file_name', 'company_name', 'standard company name', 'LLC/INC', 'ticker', 'Exchange']

ticker_map head:
                     original_patent_file_name  \
0  International Business Machines Corporation   
1          MICROSOFT TECHNOLOGY LICENSING, LLC   
2                                   Google LLC   
3                    AMAZON TECHNOLOGIES, INC.   
4                            Intel Corporation   

                                  company_name  \
0  International Business Machines Corporation   
1               MICROSOFT TECHNOLOGY LICENSING   
2                                   Google LLC   
3                          AMAZON TECHNOLOGIES   
4                            Intel Corporation   

                               standard company name LLC/INC ticker Exchange  
0  International Business Machines Corporation (IBM)     NaN    IBM     NYSE  
1                              Microsoft Corporation     LLC   MSFT   

In [ ]:
print("\nunique ticker list sample:")
print(sorted(firm_quarter_merged["ticker"].dropna().unique())[:300])


unique ticker list sample:
['AAPL', 'ABNB', 'ABT', 'ADBE', 'ADI', 'ADP', 'ADSK', 'AKAM', 'ALGN', 'ALRM', 'AMAT', 'AMBA', 'AMD', 'AMZN', 'ANET', 'AVGO', 'BA', 'BAC', 'BAX', 'BDX', 'BKR', 'BSX', 'BSY', 'CARR', 'CAT', 'CDNS', 'CGNX', 'CHTR', 'CIEN', 'CME', 'CMI', 'COF', 'CRM', 'CRUS', 'CSCO', 'CVLT', 'CVX', 'DBX', 'DE', 'DELL', 'DMRC', 'DOCU', 'DXCM', 'EA', 'EBAY', 'EGHT', 'EMR', 'EQIX', 'EXPGY', 'F', 'FARO', 'FDX', 'FTNT', 'GDDY', 'GEV ', 'GFS', 'GM', 'GMED', 'GNTX', 'GOOGL', 'GPRO', 'HOLX', 'HON', 'HPE', 'IBM', 'IDCC', 'IHRT', 'ILMN', 'INTC', 'INTU', 'IRBT', 'ISRG', 'ITW', 'JNPR', 'KD', 'KLAC', 'KN', 'LMT', 'LYFT', 'LYV', 'MASI', 'MDT', 'META', 'MS', 'MSFT', 'MSI', 'MSTR', 'MU', 'NDAQ', 'NET', 'NFLX', 'NIO', 'NKE', 'NOW', 'NTAP', 'NTNX', 'NVDA', 'ORCL', 'OWL', 'PANW', 'PATH', 'PG', 'PLTR', 'PSTG', 'PYPL', 'QCOM', 'QMCO', 'RBRK', 'RMBS', 'RNG', 'ROK', 'ROKU', 'RPD', 'RTX', 'SATS', 'SNAP', 'SNN', 'SNOW', 'SNPS', 'SONO', 'SOUN', 'SQ', 'SSTK', 'SYK', 'SYNA', 'TDC', 'TDY', 'TEAM', 'TGT', 'T